# D20 S2 A1 - Grad-CAM with NumPy
Implements the complete Grad-CAM arithmetic from feature maps and gradients.


In [ ]:
import numpy as np

fmaps = np.array([[[1., 2.], [3., 4.]],
                  [[0., 1.], [1., 0.]]])

grads = np.array([[[0., 2.], [4., 2.]],
                  [[1., 1.], [1., 5.]]])


In [ ]:
def gap_weights(grads):
    grads = np.asarray(grads)
    return grads.mean(axis=(1, 2))


def weighted_sum(fmaps, weights):
    fmaps = np.asarray(fmaps)
    weights = np.asarray(weights)
    return np.sum(fmaps * weights[:, None, None], axis=0)


def relu(x):
    x = np.asarray(x)
    return np.maximum(x, 0)


def normalize01(x):
    x = np.asarray(x, dtype=float)
    x_min = x.min()
    x_max = x.max()
    if x_max == x_min:
        return np.zeros_like(x, dtype=float)
    return (x - x_min) / (x_max - x_min)


def grad_cam(fmaps, grads):
    weights = gap_weights(grads)
    cam = weighted_sum(fmaps, weights)
    cam = relu(cam)
    return normalize01(cam)


def upsample_nearest(heat, out_h, out_w):
    heat = np.asarray(heat)
    h, w = heat.shape
    row_idx = np.arange(out_h) * h // out_h
    col_idx = np.arange(out_w) * w // out_w
    return heat[row_idx[:, None], col_idx[None, :]]


In [ ]:
w = gap_weights(grads)
print(w)
print(weighted_sum(fmaps, np.array([2., 3.])))
print(grad_cam(fmaps, grads))
print(upsample_nearest(np.array([[0., 1.], [2., 3.]]), 4, 4))


In [ ]:
# Checks corresponding to all 8 autograder requirements
assert np.allclose(gap_weights(grads), np.array([2.0, 2.0]))

expected_weighted = np.array([[2., 7.], [9., 8.]])
assert np.allclose(weighted_sum(fmaps, np.array([2., 3.])), expected_weighted)

relu_input = np.array([[-2., 1.], [3., -4.]])
assert np.array_equal(relu(relu_input), np.array([[0., 1.], [3., 0.]]))

norm_input = np.array([[2., 4.], [6., 10.]])
norm = normalize01(norm_input)
assert np.isclose(norm.min(), 0.0)
assert np.isclose(norm.max(), 1.0)

cam = grad_cam(fmaps, grads)
assert cam.shape == (2, 2)
assert np.all(cam >= 0) and np.all(cam <= 1)
expected_cam = np.array([[0., 2/3], [1., 1.]])
assert np.allclose(cam, expected_cam)

constant = np.full((3, 3), 5.0)
assert np.array_equal(normalize01(constant), np.zeros((3, 3)))

heat = np.array([[0., 1.], [2., 3.]])
expected_up = np.array([[0., 0., 1., 1.],
                        [0., 0., 1., 1.],
                        [2., 2., 3., 3.],
                        [2., 2., 3., 3.]])
assert np.array_equal(upsample_nearest(heat, 4, 4), expected_up)

print("All 8 autograder-style checks passed.")
